[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch05-nn-computation/04-huan-luyen-va-suy-luan.ipynb)

# Từ vòng lặp huấn luyện đến đường ống suy luận

Chương 5 đi hết vòng đời của mạng 784-128-64-10: huấn luyện nó, rồi đưa nó vào một đường ống suy
luận (inference) có tiền xử lý, hậu xử lý và đường chuyển cho người duyệt. Ở mỗi bước, chương đặt
một con số: khoảng 7 MOp cho một lượt truyền xuôi, 1,875 vòng cho một epoch, khoảng 554 KB khi suy
luận, 55 ms vượt ngân sách 50 ms. Sổ tay này chạy từng bước đó trên MNIST thật, kiểm lại các con số,
và đo những thứ chương để cho bạn tự đo.

**Bạn sẽ làm:**
1. đếm phép một lượt truyền xuôi, rồi ước lượng thời gian một epoch trước khi chạy;
2. viết vòng lặp minibatch và thử bốn cấu hình batch và tốc độ học (learning rate);
3. tạo quá khớp (overfitting) trên 500 ảnh và giữ checkpoint tốt nhất theo tập kiểm định;
4. dựng đường ống suy luận, đo từng bước, đo độ trễ (latency) và thông lượng (throughput) theo batch;
5. chọn ngưỡng từ chối (rejection threshold) theo chi phí, rồi xem một dịch chuyển phân phối
   (distribution shift) làm gì với nó.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import pandas as pd
import torch.nn.functional as F
import torchvision
from sklearn.datasets import load_digits

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định

tap_huan_luyen = torchvision.datasets.MNIST(DATA, train=True, download=True)
tap_kiem_tra = torchvision.datasets.MNIST(DATA, train=False, download=True)
theo_sach("số ảnh huấn luyện của MNIST", len(tap_huan_luyen), sach=60_000, nguon="60.000 ảnh huấn luyện")

N_HL, N_KD, N_KT = 10_000, 2_000, 5_000  # tập con, để cả sổ tay chạy trong khoảng một phút


def phang(anh_uint8):
    return anh_uint8.reshape(len(anh_uint8), 784).float() / 255


X_hl, y_hl = phang(tap_huan_luyen.data[:N_HL]), tap_huan_luyen.targets[:N_HL]
# tập kiểm định lấy từ phần cuối tập huấn luyện, không trùng với 10,000 ảnh đầu
X_kd, y_kd = phang(tap_huan_luyen.data[50_000:50_000 + N_KD]), tap_huan_luyen.targets[50_000:50_000 + N_KD]
X_kt, y_kt = phang(tap_kiem_tra.data[:N_KT]), tap_kiem_tra.targets[:N_KT]
print(f"huấn luyện {len(X_hl):,} · kiểm định {len(X_kd):,} · kiểm tra {len(X_kt):,} ảnh")

KICH_THUOC = (784, 128, 64, 10)


def mlp(kich_thuoc=KICH_THUOC):
    lop = []
    for vao, ra in zip(kich_thuoc[:-2], kich_thuoc[1:-1]):
        lop += [torch.nn.Linear(vao, ra), torch.nn.ReLU()]
    return torch.nn.Sequential(*lop, torch.nn.Linear(kich_thuoc[-2], kich_thuoc[-1]))


def thoi_gian(f, lap=5):
    """Trung vị thời gian của f(), sau một lần chạy khởi động."""
    f()
    do = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        do.append(time.perf_counter() - t0)
    return float(np.median(do))


def danh_gia(mo_hinh, X, y):
    """Mất mát trung bình và độ chính xác, không tính gradient."""
    with torch.no_grad():
        z = mo_hinh(X)
        return F.cross_entropy(z, y).item(), (z.argmax(1) == y).float().mean().item()

## 1. Trước khi chạy: đếm phép, rồi ước lượng một epoch

Bảng 11 của chương đếm phép cho một lượt truyền xuôi qua mạng ở batch 32. Một phép nhân ma trận
$(M \times K)(K \times N)$ tốn $2MKN$ phép, một nhân và một cộng cho mỗi số hạng; bias cộng thêm $MN$
phép, ReLU thêm $MN$ phép so sánh. Bảng dưới đây dựng lại đúng cách đếm đó.

In [ ]:
B = sach(32, nguon="32 / 64 / 128 ảnh mỗi vòng")


def bang_phep(kich_thuoc, batch):
    dong = []
    for i, (vao, ra) in enumerate(zip(kich_thuoc[:-1], kich_thuoc[1:]), 1):
        cuoi = i == len(kich_thuoc) - 1
        dong.append((f"tầng {i}", "nhân ma trận", 2 * batch * vao * ra))
        dong.append((f"tầng {i}", "bias + softmax" if cuoi else "bias + ReLU", 2 * batch * ra))
    return pd.DataFrame(dong, columns=["tầng", "phép", "số phép"])


phep = bang_phep(KICH_THUOC, B)
tong = int(phep["số phép"].sum())
tang_1 = int(phep.loc[phep["tầng"] == "tầng 1", "số phép"].sum())
nhan_ma_tran = int(phep.loc[phep["phép"] == "nhân ma trận", "số phép"].sum())
theo_sach("một lượt truyền xuôi ở batch 32 (MOp)", tong / 1e6, sach=7,
          nguon="~7 MOp mỗi lượt truyền xuôi ở batch 32", tol=0.5)
theo_sach("mỗi ảnh (KOp)", tong / B / 1e3, sach=219, nguon="~219 KOp mỗi ảnh", tol=0.5)
theo_sach("phần của tầng 1, gồm cả bias và ReLU của nó (%)", 100 * tang_1 / tong, sach=91.9, nguon="91,9 %")
print(f"riêng phép nhân ma trận của tầng 1: {100 * phep['số phép'][0] / tong:.2f} %")
MOT_PHAN_TRAM = sach(1, nguon="KOp mỗi ảnh · 1 %")
assert 100 * (tong - nhan_ma_tran) / tong < MOT_PHAN_TRAM  # bias và hàm kích hoạt: dưới 1 %
print(f"bias và hàm kích hoạt: {100 * (tong - nhan_ma_tran) / tong:.2f} % tổng số phép")
phep

Tầng 1 chiếm gần hết công việc vì nó nhân đầu vào 784 chiều với ma trận trọng số rộng nhất. Con số
91.9 % của chương khớp khi tính cả bias và ReLU của tầng 1; riêng phép nhân ma trận của nó thấp hơn
một chút, như dòng in ra ở trên.

Nhân đôi hai tầng ẩn thì sao? Tầng 1 và tầng 3 gấp đôi, tầng 2 gấp bốn, nên tổng tăng ít hơn bốn lần:

In [ ]:
gap_doi = int(bang_phep((784, 256, 128, 10), B)["số phép"].sum())
theo_sach("mạng 784-256-128-10 ở batch 32 (MOp)", gap_doi / 1e6, sach=15, nguon="~15 MOp", tol=0.5)
theo_sach("so với mạng gốc", gap_doi / tong, sach=2.15,
          trich="the total grows by about 2.15× rather than four times", tol=0.005)
print(f"{gap_doi:,} phép so với {tong:,}")

Bảng 9 của chương cho một phép nhẩm thời gian một epoch:

$$T_\text{epoch} \approx \frac{60\text{K} \times \text{FLOP xuôi và ngược mỗi ảnh}}{\text{FLOP/s đạt được}}$$

Lượt ngược của mỗi tầng tuyến tính cần hai phép nhân ma trận cùng cỡ với lượt xuôi: một cho gradient
theo trọng số, một cho gradient theo đầu vào. Tầng đầu không cần gradient theo đầu vào, vì ảnh không
có gì để cập nhật. Đếm như vậy, rồi đo FLOP/s mà CPU này đạt được với đúng các phép nhân ma trận đó:

In [ ]:
cap_tang = list(zip(KICH_THUOC[:-1], KICH_THUOC[1:]))
flop_xuoi = sum(2 * v * r for v, r in cap_tang)
flop_nguoc = sum(2 * v * r for v, r in cap_tang) + sum(2 * v * r for v, r in cap_tang[1:])
print(f"mỗi ảnh: {flop_xuoi:,} FLOP xuôi + {flop_nguoc:,} FLOP ngược = {flop_xuoi + flop_nguoc:,} FLOP")

dat_hat_giong()
W = [torch.randn(v, r) for v, r in cap_tang]
A = [torch.randn(B, v) for v, r in cap_tang]
G = [torch.randn(B, r) for v, r in cap_tang]


def chi_nhan_ma_tran():
    for a, w in zip(A, W):
        a @ w          # lượt xuôi
    for a, g in zip(A, G):
        a.T @ g        # gradient theo trọng số
    for g, w in zip(G[1:], W[1:]):
        g @ w.T        # gradient theo đầu vào, trừ tầng đầu


flop_s = B * (flop_xuoi + flop_nguoc) / thoi_gian(chi_nhan_ma_tran, lap=200)
do_tren_may("FLOP/s đạt được, chỉ phép nhân ma trận ở batch 32", flop_s / 1e9, "GFLOP/s")

VONG = math.ceil(len(tap_huan_luyen) / B)
theo_sach("số vòng mỗi epoch ở batch 32", VONG, sach=1875, nguon="1.875 vòng mỗi epoch")
uoc_luong_epoch = len(tap_huan_luyen) * (flop_xuoi + flop_nguoc) / flop_s
do_tren_may("ước lượng một epoch theo công thức của Bảng 9", uoc_luong_epoch, "s")

### Dự đoán

Phần 2 sẽ đo một vòng lặp huấn luyện thật. Nó chạy một epoch nhanh bằng ước lượng trên, chậm hơn
một chút, hay chậm hơn nhiều lần? Nếu chậm hơn, thời gian thêm vào đi đâu?

## 2. Vòng lặp minibatch

Mỗi vòng xử lý một batch qua bốn bước: truyền xuôi, tính hàm mất mát (loss function), lan truyền
ngược, cập nhật trọng số. Mỗi epoch xáo trộn dữ liệu rồi đi qua nó từng minibatch. Đây là hạ gradient
ngẫu nhiên (stochastic gradient descent, SGD) theo minibatch, viết ra đủ, không gói trong hàm thư viện
nào:

In [ ]:
def mot_epoch(mo_hinh, bo_toi_uu, X, y, batch, nhat_ky):
    thu_tu = torch.randperm(len(X))                   # xáo trộn mỗi epoch
    for i in range(0, len(X), batch):
        chon = thu_tu[i:i + batch]
        du_doan = mo_hinh(X[chon])                    # 1. truyền xuôi
        mat_mat = F.cross_entropy(du_doan, y[chon])   # 2. mất mát, lấy trung bình trên batch
        bo_toi_uu.zero_grad()
        mat_mat.backward()                            # 3. lan truyền ngược
        bo_toi_uu.step()                              # 4. cập nhật: θ ← θ − η·∇θ L
        nhat_ky.append(mat_mat.item())


dat_hat_giong()
mang_moi = mlp()
mat_mat_dau, dung_dau = danh_gia(mang_moi, X_kt, y_kt)
DOAN_MO = sach(10, nguon="10 % → 95 % độ chính xác")
MUC_TIEU = sach(95, nguon="10 % → 95 % độ chính xác")
do_tren_may("độ chính xác của mạng chưa huấn luyện", 100 * dung_dau, "%")
do_tren_may("mất mát lúc khởi đầu", mat_mat_dau)
print(f"đoán đều mười lớp: độ chính xác {DOAN_MO} %, mất mát log(10) = {math.log(10):.3f}")

Mạng vừa khởi tạo đoán không hơn gì đoán mò, và mất mát của nó nằm sát $\log 10$, mức của một phân
phối đều trên mười lớp. Chương nói học có giám sát đưa nó tới khoảng 95 %.

Giờ chạy bốn cấu hình, mỗi cấu hình 3 epoch trên 10,000 ảnh. Hai trong số đó dùng hai kích thước
batch (batch size) chương nêu, 32 và 128. Tốc độ học thì chương không cho giá trị nào:
nó nói không có mức "vừa phải" chung cho mọi trường hợp, nên phải thử trong đúng cấu hình huấn luyện.

**Dự đoán:** cấu hình nào đạt độ chính xác cao nhất sau 3 epoch? Tốc độ học 1.0 sẽ học nhanh hơn,
hay hỏng?

In [ ]:
B_LON = sach(128, nguon="32 / 64 / 128 ảnh mỗi vòng")
CAU_HINH = [(B, 0.1), (B, 0.01), (B_LON, 0.1), (B, 1.0)]
SO_EPOCH = 3

ket_qua, nhat_ky = [], {}
for batch, toc_do_hoc in CAU_HINH:
    ten = f"batch {batch}, tốc độ học {toc_do_hoc}"
    dat_hat_giong()
    m = mlp()
    bo_toi_uu = torch.optim.SGD(m.parameters(), lr=toc_do_hoc)
    nhat_ky[ten] = []
    t0 = time.perf_counter()
    for _ in range(SO_EPOCH):
        mot_epoch(m, bo_toi_uu, X_hl, y_hl, batch, nhat_ky[ten])
    giay_moi_epoch = (time.perf_counter() - t0) / SO_EPOCH
    with torch.no_grad():
        nron_chet = int((torch.relu(m[0](X_kd)).amax(0) == 0).sum())  # nơ-ron tầng 1 không bao giờ bật
    ket_qua.append({"cấu hình": ten, "độ chính xác kiểm tra (%)": 100 * danh_gia(m, X_kt, y_kt)[1],
                    "nơ-ron tầng 1 tắt hẳn (trên 128)": nron_chet, "giây mỗi epoch, 10,000 ảnh": giay_moi_epoch})
    do_tren_may(f"{ten}: độ chính xác", ket_qua[-1]["độ chính xác kiểm tra (%)"], "%")

pd.DataFrame(ket_qua).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for (ten, mm), mau in zip(nhat_ky.items(), MAU):
    batch = int(ten.split(",")[0].split()[-1])
    cua_so = max(1, 640 // batch)  # trung bình trượt trên khoảng 640 ảnh
    muot = np.convolve(mm, np.ones(cua_so) / cua_so, mode="valid")
    anh_da_xem = (np.arange(len(muot)) + cua_so) * batch / 1000
    ax.plot(anh_da_xem, muot, color=mau, lw=1.8, label=ten)
ax.axhline(math.log(10), color="0.5", lw=0.8, ls="--")
ax.text(anh_da_xem[-1], math.log(10) + 0.06, "log(10): mức của đoán đều", fontsize=8, color="0.35", ha="right")
ax.set_ylim(0, 2.8)
ax.set_xlabel("số ảnh đã xem (nghìn)")
ax.set_ylabel("mất mát huấn luyện (trung bình trượt)")
ax.set_title("Bốn cấu hình SGD, cùng 3 epoch trên 10,000 ảnh")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()
plt.show()

Ba điều thường thấy trên đồ thị và bảng, dù con số cụ thể tuỳ máy:

* tốc độ học 0.01 đi đúng hướng nhưng chậm: sau 3 epoch nó vẫn chưa tới nơi;
* batch 128 cùng tốc độ học 0.1 cập nhật ít hơn bốn lần mỗi epoch, nên sau cùng số ảnh nó thường đi
  chậm hơn, dù mỗi epoch chạy nhanh hơn vì mỗi phép nhân ma trận lớn hơn;
* tốc độ học 1.0 nhảy quá đà: mất mát dao động hoặc kẹt cao, và nhiều nơ-ron ReLU ở tầng 1 tắt hẳn,
  không bật với bất kỳ ảnh kiểm định nào, nên gradient qua chúng bằng 0 mãi mãi.

Kích thước batch vừa là một đơn vị công việc cho phần cứng, vừa quyết định nhịp cập nhật. Đó là lý
do chương nói tốc độ học, batch và cấu hình huấn luyện phải được kiểm định cùng nhau.

Còn ước lượng ở phần 1 thì sao?

In [ ]:
vong_10k = math.ceil(N_HL / B)
giay_moi_vong = ket_qua[0]["giây mỗi epoch, 10,000 ảnh"] / vong_10k
do_tren_may("thời gian mỗi vòng, batch 32", 1000 * giay_moi_vong, "ms")
do_tren_may("một epoch 60,000 ảnh: số vòng × thời gian đo mỗi vòng", VONG * giay_moi_vong, "s")
do_tren_may("so với ước lượng chỉ tính FLOP", VONG * giay_moi_vong / uoc_luong_epoch, "lần")

Công thức chỉ tính phép nhân ma trận. Vòng lặp thật còn trả cho mọi thứ khác: vòng `for` của Python,
việc ghi lại đồ thị tính toán cho autograd, bước cập nhật của bộ tối ưu (optimizer), việc chọn batch
bằng chỉ số, hàm kích hoạt (activation function), softmax, và chi phí cố định của mỗi lần gọi một
kernel nhỏ. Với một mạng nhỏ như thế này ở batch 32, phần chi phí ngoài FLOP đó có thể ngang hoặc lớn
hơn chính FLOP. Đây là lý do Bảng 9 ghi
"FLOP/s đạt được", và vì sao một phép nhẩm chỉ là điểm khởi đầu trước khi đo.

## 3. Quá khớp, tập kiểm định và checkpoint

Chương lấy ví dụ một mạng đúng 99.5 % trên dữ liệu huấn luyện mà chỉ đúng 85 % trên dữ liệu chưa
thấy: một khoảng cách tổng quát hoá lớn, dấu hiệu mạnh của quá khớp. Cách dễ nhất để
tạo ra nó là cho mạng học thuộc một tập nhỏ: 500 ảnh, 40 epoch.

Trong lúc chạy, mỗi epoch đánh giá trên tập kiểm định, và mỗi khi mất mát kiểm định thấp hơn mọi lần
trước thì lưu trọng số xuống đĩa. Chương gọi việc này là checkpointing.

In [ ]:
VI_DU_HL = sach(99.5, nguon="99,5 % trên tập huấn luyện")
VI_DU_MOI = sach(85, nguon="85 % trên dữ liệu chưa thấy")
print(f"ví dụ của chương: {VI_DU_HL} % so với {VI_DU_MOI} %, khoảng cách {VI_DU_HL - VI_DU_MOI:.1f} điểm")

N_NHO, EPOCH_NHO = 500, 40
TEP = "checkpoint_tot_nhat.pt"  # ghi trong thư mục làm việc
dat_hat_giong()
m_nho = mlp()
bo_toi_uu = torch.optim.Adam(m_nho.parameters(), lr=1e-3)
lich_su, tot_nhat = [], float("inf")
for ep in range(1, EPOCH_NHO + 1):
    mot_epoch(m_nho, bo_toi_uu, X_hl[:N_NHO], y_hl[:N_NHO], B, [])
    lich_su.append((*danh_gia(m_nho, X_hl[:N_NHO], y_hl[:N_NHO]), *danh_gia(m_nho, X_kd, y_kd)))
    if lich_su[-1][2] < tot_nhat:
        tot_nhat, ep_tot = lich_su[-1][2], ep
        t0 = time.perf_counter()
        torch.save(m_nho.state_dict(), TEP)
        giay_ghi = time.perf_counter() - t0

mm_hl, dung_hl, mm_kd, dung_kd = map(np.array, zip(*lich_su))
do_tren_may("cuối cùng, độ chính xác trên 500 ảnh huấn luyện", 100 * dung_hl[-1], "%")
do_tren_may("cuối cùng, độ chính xác trên tập kiểm định", 100 * dung_kd[-1], "%")
do_tren_may("epoch có mất mát kiểm định thấp nhất", ep_tot)
do_tren_may("ghi checkpoint lần cuối", 1000 * giay_ghi, "ms")
print(f"tệp checkpoint: {os.path.getsize(TEP) / 1000:.1f} KB")

# nạp lại checkpoint tốt nhất: nó phải cho đúng mất mát kiểm định đã ghi nhận
m_lai = mlp()
m_lai.load_state_dict(torch.load(TEP, weights_only=True))
assert abs(danh_gia(m_lai, X_kd, y_kd)[0] - tot_nhat) < 1e-6
os.remove(TEP)

In [ ]:
epoch = np.arange(1, EPOCH_NHO + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(epoch, 100 * dung_hl, color=MAU[0], lw=2, label="500 ảnh huấn luyện")
ax1.plot(epoch, 100 * dung_kd, color=MAU[1], lw=2, label="tập kiểm định")
ax1.fill_between(epoch, 100 * dung_kd, 100 * dung_hl, color="0.85", alpha=0.6, lw=0)
ax1.text(EPOCH_NHO * 0.55, (100 * dung_kd[-1] + 100 * dung_hl[-1]) / 2, "khoảng cách\ntổng quát hoá",
         fontsize=8, color="0.3", ha="center", va="center")
ax1.set_ylim(min(100 * dung_kd.min(), 60) - 2, 101)
ax1.set_xlabel("epoch")
ax1.set_ylabel("độ chính xác (%)")
ax1.set_title("Độ chính xác")
ax1.legend(loc="lower right", fontsize=8)
ax2.plot(epoch, mm_hl, color=MAU[0], lw=2, label="500 ảnh huấn luyện")
ax2.plot(epoch, mm_kd, color=MAU[1], lw=2, label="tập kiểm định")
ax2.axvline(ep_tot, color="0.4", lw=1, ls="--")
ax2.annotate(f"checkpoint tốt nhất: epoch {ep_tot}", (ep_tot, tot_nhat), xytext=(12, 40),
             textcoords="offset points", fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax2.set_yscale("log")
ax2.set_xlabel("epoch")
ax2.set_ylabel("mất mát (thang log)")
ax2.set_title("Mất mát")
ax2.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

Mất mát huấn luyện cứ giảm, còn mất mát kiểm định đến một lúc thì dừng giảm rồi đi lên: đúng dấu hiệu
chương mô tả. Checkpoint giữ lại trọng số ở điểm tốt nhất đó, nên việc chạy thêm không làm mất nó. Với
mạng MNIST, ghi một checkpoint gần như không tốn thời gian; với mô hình có tham số chiếm hàng chục tới
hàng trăm gigabyte, chính việc ghi đó có thể làm vòng lặp huấn luyện phải dừng chờ.

## 4. Mô hình cho phần còn lại, và trọng số tầng đầu

Phần còn lại của sổ tay cần một mạng tốt hơn: Adam, 5 epoch trên 10,000 ảnh. Sau đó trọng số được
đóng băng; suy luận không cập nhật chúng nữa.

In [ ]:
dat_hat_giong()
mang = mlp()
bo_toi_uu = torch.optim.Adam(mang.parameters(), lr=1e-3)
for _ in range(5):
    mot_epoch(mang, bo_toi_uu, X_hl, y_hl, B, [])
mang.eval()
for p in mang.parameters():
    p.requires_grad_(False)  # đóng băng
tham_so = sum(p.numel() for p in mang.parameters())
dung_mnist = danh_gia(mang, X_kt, y_kt)[1]
do_tren_may("độ chính xác trên tập kiểm tra", 100 * dung_mnist, "%")

Một ngộ nhận chương bác bỏ là mạng nơ-ron hoàn toàn mờ đục, không gỡ lỗi được. Mạng không cho một vết
nhân quả từng dòng như code thường, nhưng trọng số tầng đầu của bộ phân loại MNIST vẫn là một phép
kiểm tra có ích. Mỗi nơ-ron của tầng 1 có 784 trọng số, đúng một ảnh 28×28:

In [ ]:
W1 = mang[0].weight.numpy()
with torch.no_grad():
    tat_han = (torch.relu(mang[0](X_kd)).amax(0) == 0).numpy()  # không bật với ảnh kiểm định nào
do_tren_may("nơ-ron tầng 1 tắt hẳn trong mô hình này (trên 128)", int(tat_han.sum()))
fig, axes = plt.subplots(3, 8, figsize=(10, 4.4))
for k, ax in enumerate(axes.flat):
    gioi_han = np.abs(W1[k]).max()
    ax.imshow(W1[k].reshape(28, 28), cmap="RdBu_r", vmin=-gioi_han, vmax=gioi_han)
    if tat_han[k]:
        ax.set_title("tắt hẳn", fontsize=8, color="0.3")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
fig.suptitle("24 trong 128 nơ-ron tầng 1: đỏ là trọng số dương, xanh là âm, trắng là gần 0", fontsize=10)
plt.tight_layout()
plt.show()

Ở giữa khung là những mảng đỏ và xanh xếp thành nét: mỗi nơ-ron nhạy với mực ở chỗ này và với khoảng
trống ở chỗ kia. Vùng viền chỉ có nhiễu nhạt, vì những pixel đó gần như luôn trống, gradient của chúng
bằng 0, và trọng số của chúng giữ nguyên giá trị khởi tạo ngẫu nhiên. Một nơ-ron mà cả khung chỉ toàn
nhiễu thì gần như chưa học gì; nếu nó được đánh dấu "tắt hẳn", nó không bật với ảnh nào, nên không có
gradient nào tới được nó.

Đây là bằng chứng một phần, chưa phải lời giải thích đầy đủ, đúng như chương dè dặt. Nhưng nó cho biết
mạng đang nhìn vào đâu, và đếm được bao nhiêu nơ-ron đang bị bỏ phí: một phép gỡ lỗi thật.

## 5. Đường ống suy luận

### Bộ nhớ khi suy luận

Khi suy luận, tham số phải luôn sẵn sàng, còn giá trị kích hoạt (activation) của mỗi tầng chỉ cần
sống tới khi tầng sau dùng xong, nên vài bộ đệm (buffer) dùng xoay vòng là đủ. Một ảnh tạo ra bao
nhiêu giá trị kích hoạt?

In [ ]:
gia_tri_tang = sum(KICH_THUOC[1:])
theo_sach("giá trị kích hoạt các tầng tạo ra, một ảnh", gia_tri_tang, sach=202, nguon="202 giá trị kích hoạt")
theo_sach("kể cả bộ đệm đầu vào", gia_tri_tang + KICH_THUOC[0], sach=986, nguon="986 kể cả đệm đầu vào")
theo_sach("ở batch 32", B * gia_tri_tang, sach=6464, nguon="6.464 ở batch 32")

# suy luận chỉ cần hai bộ đệm kề nhau: đầu vào của tầng đang chạy và đầu ra của nó
cap_lon_nhat = max(v + r for v, r in cap_tang)
kb_tham_so = tham_so * 4 / 1000
kb_kich_hoat = B * cap_lon_nhat * 4 / 1000
theo_sach("bộ đệm kích hoạt khi suy luận, batch 32 (KB)", kb_kich_hoat, sach=117, nguon="437,5 KB · 117 KB")
kb_suy_luan = kb_tham_so + kb_kich_hoat
theo_sach("tổng khi suy luận (KB)", kb_suy_luan, sach=554, nguon="117 KB · ~554 KB", tol=1)

# huấn luyện: tham số, gradient, hai bộ đệm Adam, và mọi giá trị kích hoạt giữ lại cho lượt ngược
kb_huan_luyen = 4 * kb_tham_so + B * (gia_tri_tang + KICH_THUOC[0]) * 4 / 1000
theo_sach("huấn luyện so với suy luận, cùng batch 32", kb_huan_luyen / kb_suy_luan, sach=3.4,
          nguon="3,4× · 13.712,9×")
print(f"huấn luyện {kb_huan_luyen:,.1f} KB · suy luận {kb_suy_luan:,.1f} KB")

### Bốn bước, mỗi bước một hình dạng chi phí

Một ảnh chữ số thật không đến dưới dạng vectơ 784 số đã chuẩn hoá. Đường ống ở đây có ba bước:

1. **Tiền xử lý:** cắt khung bao nét chữ, co giãn nó vào một ô 20×20 giữ nguyên tỉ lệ, đặt vào giữa
   khung 28×28, rồi dời trọng tâm nét chữ vào giữa khung, giống cách ảnh MNIST đã được chuẩn hoá.
2. **Truyền xuôi** qua mạng đã đóng băng.
3. **Hậu xử lý:** softmax, lấy chữ số có xác suất cao nhất, so độ tin cậy với ngưỡng, và đóng gói kết
   quả cho hệ phía sau, kể cả cờ "chuyển cho người duyệt".

In [ ]:
TRUC = torch.arange(28, dtype=torch.float32)


def tien_xu_ly(anh, o=20):
    """Một ảnh xám bất kỳ cỡ, giá trị 0..1, thành khung 28×28 kiểu MNIST."""
    muc = anh > 0.1
    hang, cot = muc.any(1).nonzero().flatten(), muc.any(0).nonzero().flatten()
    if len(hang) == 0:
        return torch.zeros(28, 28)
    cat = anh[hang[0]:hang[-1] + 1, cot[0]:cot[-1] + 1]                            # cắt khung bao
    ti_le = o / max(cat.shape)
    cao, rong = (max(1, round(n * ti_le)) for n in cat.shape)
    nho = F.interpolate(cat[None, None], size=(cao, rong), mode="bilinear", align_corners=False)[0, 0]
    khung = torch.zeros(28, 28)
    y0, x0 = (28 - cao) // 2, (28 - rong) // 2
    khung[y0:y0 + cao, x0:x0 + rong] = nho.clamp(0, 1)                               # vào ô 20×20, giữa khung
    tong_muc = khung.sum()
    cy, cx = (khung.sum(1) * TRUC).sum() / tong_muc, (khung.sum(0) * TRUC).sum() / tong_muc
    return torch.roll(khung, (round(13.5 - cy.item()), round(13.5 - cx.item())), (0, 1))  # dời trọng tâm


def hau_xu_ly(logit, nguong):
    xac_suat = torch.softmax(logit, 1)
    tin_cay, nhan = xac_suat.max(1)
    return [{"chu_so": n, "tin_cay": round(c, 3), "chuyen_nguoi_duyet": c < nguong}
            for c, n in zip(tin_cay.tolist(), nhan.tolist())]


def duong_ong(anh_tho, nguong=0.9):
    X = torch.stack([tien_xu_ly(a) for a in anh_tho]).reshape(len(anh_tho), 784)
    with torch.no_grad():
        return hau_xu_ly(mang(X), nguong)


anh_tho = tap_kiem_tra.data[:N_KT].float() / 255
print(duong_ong(anh_tho[:3]))
X_qua_ong = torch.stack([tien_xu_ly(a) for a in anh_tho]).reshape(-1, 784)
do_tren_may("độ chính xác trên ảnh kiểm tra đi qua bước tiền xử lý", 100 * danh_gia(mang, X_qua_ong, y_kt)[1], "%")

Ngay trên ảnh MNIST, đi qua bước tiền xử lý này có thể làm độ chính xác giảm đôi chút: bước tiền xử lý
khi phục vụ không giống hệt bước đã tạo ra ảnh huấn luyện, và mạng nhạy với cả khác biệt nhỏ đó.

### Dự đoán

Với một ảnh mỗi lần, bước nào tốn thời gian nhất: tiền xử lý, truyền xuôi qua 109,184 MAC, hay hậu
xử lý? Khi gom 256 ảnh một lần thì thứ tự đó có đổi không?

In [ ]:
def thoi_gian_buoc(b, lap):
    lo = anh_tho[:b]
    t_tien = thoi_gian(lambda: torch.stack([tien_xu_ly(a) for a in lo]), lap)
    X = torch.stack([tien_xu_ly(a) for a in lo]).reshape(b, 784)
    with torch.no_grad():
        t_xuoi = thoi_gian(lambda: mang(X), lap)
        z = mang(X)
    t_hau = thoi_gian(lambda: hau_xu_ly(z, 0.9), lap)
    return 1e6 * np.array([t_tien, t_xuoi, t_hau]) / b  # µs mỗi ảnh


BUOC = ["tiền xử lý", "truyền xuôi", "hậu xử lý"]
moi_anh = {1: thoi_gian_buoc(1, 50), 256: thoi_gian_buoc(256, 5)}
for b, t in moi_anh.items():
    for ten, gt in zip(BUOC, t):
        do_tren_may(f"batch {b}, {ten}, mỗi ảnh", gt, "µs")
    do_tren_may(f"batch {b}, phần của truyền xuôi trong cả đường ống", 100 * t[1] / t.sum(), "%")

fig, ax = plt.subplots(figsize=(8, 2.8))
for j, b in enumerate(moi_anh):
    trai = 0
    for i, (ten, gt) in enumerate(zip(BUOC, moi_anh[b])):
        ax.barh(j, gt, left=trai, color=MAU[i], height=0.55, edgecolor="white", lw=2, label=ten if j == 0 else None)
        trai += gt
ax.set_yticks([0, 1], [f"batch {b}" for b in moi_anh])
ax.invert_yaxis()
ax.set_xlabel("µs mỗi ảnh (đo trên máy này)")
ax.set_title("Thời gian mỗi ảnh qua từng bước của đường ống")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

Truyền xuôi thường chỉ là một phần nhỏ: phép nhân ma trận nhanh, còn tiền xử lý và hậu xử lý chạy
tuần tự, từng ảnh một, bằng mã Python và các phép nhỏ. Gom batch làm truyền xuôi rẻ đi trên mỗi ảnh,
nhưng không giúp được vòng lặp tiền xử lý. Chương nêu đúng điều này: nếu bỏ qua mã bao quanh mô hình,
tiền xử lý và hậu xử lý có thể chiếm phần lớn độ trễ đầu-cuối dù bản thân mạng chạy nhanh.

Cạm bẫy chương nêu ở mục ngộ nhận là cùng một bài toán cộng, ở quy mô của một hệ thật:

In [ ]:
SUY_LUAN_MS = sach(20, nguon="20 ms")
TIEN_XU_LY_MS = sach(25, nguon="25 ms")
HAU_XU_LY_MS = sach(10, nguon="10 ms")
NGAN_SACH_MS = sach(50, nguon="50 ms")
dau_cuoi_ms = SUY_LUAN_MS + TIEN_XU_LY_MS + HAU_XU_LY_MS
theo_sach("độ trễ đầu-cuối trong ví dụ của chương (ms)", dau_cuoi_ms, sach=55, nguon="55 ms")
assert dau_cuoi_ms > NGAN_SACH_MS  # mô hình 20 ms vẫn làm cả hệ trượt ngân sách 50 ms
print(f"mô hình chỉ chiếm {100 * SUY_LUAN_MS / dau_cuoi_ms:.0f} % của {dau_cuoi_ms} ms, "
      f"vượt ngân sách {NGAN_SACH_MS} ms thêm {dau_cuoi_ms - NGAN_SACH_MS} ms")

### Độ trễ và thông lượng theo kích thước batch

Xử lý từng ảnh một cho độ trễ thấp nhất, gom batch cho thông lượng cao hơn vì phần cứng
song song được dùng tốt hơn. Đây cũng là câu hỏi ở phần C của lab đi kèm chương.

**Dự đoán:** khi batch tăng từ 1 lên 1024, độ trễ của một batch tăng hay giảm? Thông lượng thì sao?

In [ ]:
cac_batch = [1, 4, 16, 64, 256, 1024]
tre_ms, thong_luong = [], []
with torch.no_grad():
    for b in cac_batch:
        t = thoi_gian(lambda: mang(X_kt[:b]), lap=50 if b <= 64 else 10)
        tre_ms.append(1000 * t)
        thong_luong.append(b / t)
        do_tren_may(f"batch {b:>4}: độ trễ một batch", tre_ms[-1], "ms")
for b in (1, B, 1024):
    print(f"bộ đệm kích hoạt khi suy luận ở batch {b:>4}: {b * cap_lon_nhat * 4 / 1000:,.1f} KB")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.loglog(cac_batch, tre_ms, "o-", color=MAU[0], lw=2)
ax1.set_xlabel("kích thước batch")
ax1.set_ylabel("ms cho cả batch")
ax1.set_title("Độ trễ của một batch")
ax2.semilogx(cac_batch, np.array(thong_luong) / 1000, "o-", color=MAU[2], lw=2)
ax2.set_ylim(0, max(thong_luong) / 1000 * 1.15)
ax2.set_xlabel("kích thước batch")
ax2.set_ylabel("nghìn ảnh mỗi giây")
ax2.set_title("Thông lượng")
for ax in (ax1, ax2):
    ax.set_xticks(cac_batch, [str(b) for b in cac_batch])
fig.suptitle("Truyền xuôi của mạng 784-128-64-10, đo trên máy này", fontsize=10)
plt.tight_layout()
plt.show()

Cả hai cùng tăng: một batch lớn mất lâu hơn một ảnh, nhưng mỗi giây xử lý được nhiều ảnh hơn hẳn, cho
tới khi phần cứng đã bận hết. Một dịch vụ thời gian thực chọn batch nhỏ để giữ độ trễ, một tác vụ xử lý
theo lô chọn batch lớn để lấy thông lượng. Còn bộ đệm kích hoạt thì lớn lên tuyến tính theo batch.

## 6. Ngưỡng từ chối: chia việc giữa máy và người

Đầu ra của mạng chỉ dùng được sau khi đổi thành quyết định. Hệ nhận dạng chữ số mà chương kể không ép
mọi ảnh qua mô hình: ảnh có độ tin cậy thấp được chuyển cho người duyệt. Ngưỡng từ chối càng cao
thì càng nhiều ảnh phải chuyển, nhưng càng ít lỗi lọt qua.

In [ ]:
with torch.no_grad():
    tin_cay, du_doan = torch.softmax(mang(X_kt), 1).max(1)
sai = (du_doan != y_kt).numpy()
tin_cay = tin_cay.numpy()

cac_nguong = np.concatenate([[0.0], np.quantile(tin_cay, np.linspace(0.01, 0.6, 120))])
tu_choi = np.array([(tin_cay < t).mean() for t in cac_nguong])
loi_phan_nhan = np.array([sai[tin_cay >= t].mean() for t in cac_nguong])

LOI_1989 = sach(1, nguon="1 % lỗi sau khi từ chối")
TU_CHOI_1989 = sach(12.1, nguon="từ chối 12,1 % chữ số")
can = tu_choi[np.argmax(loi_phan_nhan <= LOI_1989 / 100)] if (loi_phan_nhan <= LOI_1989 / 100).any() else float("nan")
do_tren_may("lỗi khi không từ chối ảnh nào", 100 * loi_phan_nhan[0], "%")
do_tren_may(f"tỉ lệ phải từ chối để lỗi trên phần được nhận còn {LOI_1989} %", 100 * can, "%")
print(f"hệ 1989 của chương: {LOI_1989} % lỗi sau khi từ chối {TU_CHOI_1989} % chữ số "
      "(hệ khác, dữ liệu khác: chỉ để biết đường cong này được dùng thế nào)")

Chọn ngưỡng nào? Chương đưa một quy tắc chi phí kỳ vọng: chọn ngưỡng $t$ làm nhỏ nhất

$$\text{tỉ lệ chuyển người}(t) \cdot C_\text{người} + \text{tỉ lệ lỗi}(t) \cdot C_\text{lỗi},$$

trong giới hạn thông lượng và độ trễ. Hai chi phí là của nghiệp vụ, không phải của mô hình. Sổ tay này
**giả định** một lỗi lọt qua tốn gấp 20 lần một lần chuyển cho người.

In [ ]:
C_NGUOI, C_LOI = 1.0, 20.0  # giả định, đơn vị chi phí tuỳ ý
ty_le_loi = np.array([(sai & (tin_cay >= t)).mean() for t in cac_nguong])  # lỗi lọt qua, trên tổng số ảnh
chi_phi = tu_choi * C_NGUOI + ty_le_loi * C_LOI
i_tot = int(np.argmin(chi_phi))
NGUONG_TOT = float(cac_nguong[i_tot])
do_tren_may("ngưỡng có chi phí kỳ vọng thấp nhất", NGUONG_TOT)
do_tren_may("ở ngưỡng đó, tỉ lệ chuyển cho người", 100 * tu_choi[i_tot], "%")
do_tren_may("ở ngưỡng đó, lỗi trên phần máy nhận", 100 * loi_phan_nhan[i_tot], "%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(100 * tu_choi, 100 * loi_phan_nhan, color=MAU[0], lw=2)
ax1.axhline(LOI_1989, color="0.5", lw=0.8, ls="--")
ax1.text(100 * tu_choi.max() * 0.98, LOI_1989 + 0.12, f"lỗi {LOI_1989} %", fontsize=8, color="0.35", ha="right")
ax1.plot([100 * tu_choi[i_tot]], [100 * loi_phan_nhan[i_tot]], "o", color=MAU[1], ms=8)
ax1.annotate("ngưỡng chi phí thấp nhất", (100 * tu_choi[i_tot], 100 * loi_phan_nhan[i_tot]), xytext=(14, 18),
             textcoords="offset points", fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax1.set_xlabel("ảnh chuyển cho người (%)")
ax1.set_ylabel("lỗi trên phần máy nhận (%)")
ax1.set_title("Từ chối càng nhiều, lỗi lọt qua càng ít")
ax2.plot(cac_nguong, tu_choi * C_NGUOI, color=MAU[2], lw=1.5, label="chi phí chuyển người")
ax2.plot(cac_nguong, ty_le_loi * C_LOI, color=MAU[3], lw=1.5, label="chi phí lỗi lọt qua")
ax2.plot(cac_nguong, chi_phi, color=MAU[1], lw=2.2, label="tổng")
ax2.plot([NGUONG_TOT], [chi_phi[i_tot]], "o", color=MAU[1], ms=8)
ax2.set_xlabel("ngưỡng độ tin cậy t")
ax2.set_ylabel("chi phí kỳ vọng mỗi ảnh")
ax2.set_title(f"Chi phí, giả định C_lỗi = {C_LOI:.0f} × C_người")
ax2.legend(loc="center left", fontsize=8)
plt.tight_layout()
plt.show()

Đổi chi phí là đổi ngưỡng, không cần đổi mô hình. Vì thế chương nói ngưỡng phải phản ánh kinh tế của
việc vận hành, và đó là chỗ cần hiểu biết về lĩnh vực mà một mạng sâu không thay được. Bảng 14 của
chương còn cho một bài học nữa từ hệ 1989, ở thông lượng:

In [ ]:
CHAM, NHANH = sach(10, nguon="10–12 phân loại mỗi giây"), sach(12, nguon="10–12 phân loại mỗi giây")
SAU_CHUAN_HOA = sach(30, nguon="hơn 30 sau chuẩn hoá")
ms_dau_cuoi = (1000 / NHANH, 1000 / CHAM)
ms_phan_loai = 1000 / SAU_CHUAN_HOA
print(f"kể cả thu ảnh: {ms_dau_cuoi[0]:.0f}–{ms_dau_cuoi[1]:.0f} ms mỗi chữ số")
print(f"chỉ phân loại ảnh đã chuẩn hoá: dưới {ms_phan_loai:.0f} ms mỗi chữ số")
print(f"→ ít nhất {100 * (1 - ms_phan_loai / ms_dau_cuoi[0]):.0f} % thời gian mỗi chữ số nằm ngoài bộ phân loại")

THAM_SO_1989 = sach(10_000, nguon="~10.000 tham số", tol=1000)
print(f"mạng 1989 có khoảng {THAM_SO_1989:,} tham số, mạng của chương {tham_so:,}: "
      f"gấp khoảng {tham_so / THAM_SO_1989:.0f} lần, nhưng số tham số không tự nói gì về độ trễ")

## 7. Thế giới đổi, mô hình đứng yên

Chương mở mục suy luận bằng một tình huống: mô hình đạt 99 % trên tập kiểm tra, ba tháng sau khi triển
khai bắt đầu cho kết quả vô nghĩa, dù không ai sửa dòng code nào. Trọng số đã đóng băng; một nguyên nhân
có thể là thế giới vận hành đã dịch chuyển.

Ở đây, một dịch chuyển phân phối có sẵn trong scikit-learn: `load_digits` là
1,797 chữ số viết tay khác, chụp ở 8×8 pixel, 17 mức xám. Cùng mười chữ số, cùng mạng đóng băng, chỉ
khác nguồn ảnh.

**Dự đoán:** mạng sẽ đúng bao nhiêu trên những ảnh này nếu chỉ phóng to chúng lên 28×28? Còn nếu cho
chúng đi qua bước tiền xử lý kiểu MNIST?

In [ ]:
TINH_HUONG = sach(99, nguon="99 % · ba tháng")
chu_so = load_digits()
anh_8x8 = torch.tensor(chu_so.images, dtype=torch.float32) / 16
y_8x8 = torch.tensor(chu_so.target)
phong_to = F.interpolate(anh_8x8[:, None], size=(28, 28), mode="bilinear", align_corners=False)[:, 0]
qua_ong = torch.stack([tien_xu_ly(a) for a in anh_8x8])

nguon_anh = {"MNIST, tập kiểm tra": (X_kt, y_kt),
             "load_digits, chỉ phóng to": (phong_to.reshape(-1, 784), y_8x8),
             "load_digits, qua tiền xử lý": (qua_ong.reshape(-1, 784), y_8x8)}
bang_dich = []
with torch.no_grad():
    for ten, (X, y) in nguon_anh.items():
        tc, dd = torch.softmax(mang(X), 1).max(1)
        bang_dich.append({"nguồn ảnh": ten, "độ chính xác (%)": 100 * (dd == y).float().mean().item(),
                          f"chuyển cho người ở t = {NGUONG_TOT:.2f} (%)": 100 * (tc < NGUONG_TOT).float().mean().item(),
                          "lỗi trên phần máy nhận (%)": 100 * (dd != y)[tc >= NGUONG_TOT].float().mean().item(),
                          "độ tin cậy trung bình": tc.mean().item()})
        do_tren_may(f"{ten}: độ chính xác", bang_dich[-1]["độ chính xác (%)"], "%")
bang_dich = pd.DataFrame(bang_dich)
bang_dich.round(3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(11, 2.9), gridspec_kw={"width_ratios": [1, 1, 1, 2.2]})
for ax, (ten, anh) in zip(axes[:3], [("MNIST", X_kt[0].reshape(28, 28)), ("load_digits, phóng to", phong_to[0]),
                                     ("load_digits, qua tiền xử lý", qua_ong[0])]):
    ax.imshow(anh, cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(ten, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
ten_ngan = ["MNIST", "phóng to", "qua tiền\nxử lý"]
cot_tu_choi = bang_dich.columns[2]
x = np.arange(3)
axes[3].bar(x - 0.2, bang_dich["độ chính xác (%)"], width=0.38, color=MAU[0], label="độ chính xác")
axes[3].bar(x + 0.2, bang_dich[cot_tu_choi], width=0.38, color=MAU[1], label="chuyển cho người")
axes[3].set_xticks(x, ten_ngan, fontsize=8)
axes[3].set_ylim(0, 115)
axes[3].set_ylabel("%")
axes[3].legend(loc="upper right", fontsize=8, ncol=2)
axes[3].set_title("Cùng mạng, ba nguồn ảnh", fontsize=9)
plt.tight_layout()
plt.show()

Mạng không hề đổi, vậy mà độ chính xác rơi mạnh khi chỉ phóng to ảnh: nét chữ dày, chiếm cả khung,
không giống thứ mạng từng thấy. Tiền xử lý kéo lại được một phần, nhưng không phải tất cả, vì nó không
tạo ra được những kiểu viết mà dữ liệu huấn luyện không có.

Có một tín hiệu dùng được mà không cần nhãn: tỉ lệ chuyển cho người tăng mạnh. Chương dặn theo dõi đúng
thứ đó trong vận hành, cùng chất lượng ảnh đầu vào. Khi chưa có nhãn của dữ liệu mới, một tỉ lệ từ chối
bất thường là cảnh báo sớm nhất.

Nhưng ngưỡng không còn bảo vệ được phần ảnh nó cho qua. Hãy nhìn cột lỗi trên phần máy nhận: trên ảnh
dịch chuyển, mạng vẫn rất tự tin với nhiều câu trả lời sai. Độ tin cậy chỉ đáng tin trong vùng dữ liệu
đã dùng để kiểm định nó, nên một ngưỡng chọn trên MNIST không nói gì về mức lỗi trên nguồn ảnh khác.

### Chẩn đoán theo ba trục

Chương đề xuất một quy trình chẩn đoán, gọi là phân loại Dữ liệu, Thuật toán, Máy (D·A·M taxonomy): bắt
đầu từ một triệu chứng, đặt giả thuyết trên cả ba trục, rồi đo đại lượng phân biệt được chúng.

In [ ]:
with torch.no_grad():
    lan_1, lan_2 = mang(X_kt), mang(X_kt)
assert torch.equal(lan_1, lan_2)  # cùng trọng số, cùng FP32: lặp lại từng bit
hang = bang_dich.set_index("nguồn ảnh")
chan_doan = pd.DataFrame([
    {"trục": "Dữ liệu", "giả thuyết": "ảnh mới khác phân phối huấn luyện",
     "đo": "tỉ lệ chuyển người, độ tin cậy trung bình trên ảnh mới",
     "ở đây": f"chuyển người {hang[cot_tu_choi].iloc[1]:.0f} % so với {hang[cot_tu_choi].iloc[0]:.0f} % trên MNIST"},
    {"trục": "Thuật toán", "giả thuyết": "mô hình không đủ năng lực cho bài toán",
     "đo": "độ chính xác trên tập kiểm tra giữ riêng của MNIST",
     "ở đây": f"{100 * dung_mnist:.1f} %, không đổi: trọng số đã đóng băng"},
    {"trục": "Máy", "giả thuyết": "phần cứng hay độ chính xác số làm lệch kết quả",
     "đo": "chạy lại cùng đầu vào, cùng trọng số, cùng FP32",
     "ở đây": "đầu ra giống hệt từng bit"},
])
chan_doan

Triệu chứng nằm ở trục Dữ liệu, nên can thiệp đầu tiên cũng ở đó: tiền xử lý tốt hơn, rồi thu thập và
gán nhãn ảnh từ nguồn mới để huấn luyện lại. Chương lưu ý một can thiệp trên một trục có thể dời điểm
nghẽn sang trục khác, nên phép thử cuối cùng luôn là hành vi đầu-cuối dưới đúng tải công việc
(workload) định phục vụ.

## 8. Ngộ nhận cuối: độ chính xác 99 % chẳng bắt được gì

Chương kể một bộ phát hiện gian lận trên dữ liệu lệch 99:1. Không cần huấn luyện gì để thấy vấn đề:

In [ ]:
THUONG_MOI_GIAN_LAN = sach(99, nguon="99:1 → 99 %")
nhan_giao_dich = np.array([0] * (THUONG_MOI_GIAN_LAN * 100) + [1] * 100)  # 1 là gian lận
luon_noi_khong = np.zeros_like(nhan_giao_dich)
theo_sach("độ chính xác của bộ luôn đoán 'không gian lận' (%)", 100 * (luon_noi_khong == nhan_giao_dich).mean(),
          sach=99, nguon="99:1 → 99 %")
bat_duoc = int(((luon_noi_khong == 1) & (nhan_giao_dich == 1)).sum())
assert bat_duoc == 0
print(f"bắt được {bat_duoc} trên {int(nhan_giao_dich.sum())} vụ gian lận · "
      f"tỉ lệ âm tính giả (false negative rate): {100 * (1 - bat_duoc / nhan_giao_dich.sum()):.0f} %")

Hàm mục tiêu lấy trung bình đều trên mọi mẫu, nên lớp hiếm gần như không có tiếng nói. Chương nói phải
xử lý từ phía dữ liệu và cách đánh giá: lấy mẫu, gán trọng số, thiết kế hàm mất mát, và đo riêng từng
lớp. Một con số độ chính xác tổng không thay được những thước đo đó.

## Thử thêm

1. Ở phần 2, thêm cấu hình `(B, 0.3)` hoặc `(64, 0.1)` vào `CAU_HINH`. Số nơ-ron tắt hẳn và độ chính xác
   đổi thế nào? Nhớ rằng chương nói không có tốc độ học "vừa phải" chung.
2. Ở phần 6, đổi `C_LOI` thành 5 rồi 100. Ngưỡng tốt nhất dịch đi đâu, và bao nhiêu ảnh phải chuyển
   cho người?
3. Ở phần 5, bỏ bước dời trọng tâm trong `tien_xu_ly` (trả về `khung` thay vì `torch.roll(...)`), rồi
   chạy lại phần 7. Độ chính xác trên `load_digits` qua tiền xử lý còn bao nhiêu?

## Tóm lại

* Một lượt truyền xuôi ở batch 32 tốn khoảng 7 MOp, gần hết ở tầng 1; một phép nhẩm chỉ tính FLOP
  thường đánh giá thấp thời gian của một vòng lặp huấn luyện nhỏ, vì phần chi phí ngoài FLOP lớn.
* Kích thước batch và tốc độ học phải được thử cùng nhau; một tốc độ học quá lớn có thể tắt hẳn nhiều
  nơ-ron ReLU.
* Khoảng cách giữa tập huấn luyện và tập kiểm định là dấu hiệu quá khớp; checkpoint giữ lại điểm tốt
  nhất.
* Suy luận cần khoảng 554 KB so với khoảng 1.9 MB khi huấn luyện ở cùng batch, và độ trễ đầu-cuối gồm
  cả tiền xử lý và hậu xử lý, không chỉ mô hình.
* Ngưỡng từ chối là một quyết định kinh tế; khi dữ liệu dịch chuyển, tỉ lệ chuyển cho người là tín hiệu
  sớm nhất.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Tính toán nơ-ron](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch05-nn-computation/), dựng từ chương
[*Neural Computation*](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_05_nn_compute.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.